# Capstone Teaching Artifact v22 — Shakedown Edition

**Checkpoint:** `checkpoints/exp6_capstone_seed0_step2000.pt` (MP-88 shakedown, 2000 steps, seed 0)
**Config:** Joint modular (P=113, 30% train) + induction (vocab=2048, seq_len=128); d_model=256, 4 layers, 8 heads
**Manifest:** `results/probe_capstone_shakedown.json` (git_sha db9c4c6, clean)

**Honest framing:** This config finds a dense algorithm for modular addition and compositional induction without a confirmed head. The dissociation is the finding.

In [ ]:
# Cell 1: Load capstone model + config from checkpoint
import torch
import sys
sys.path.insert(0, 'src')

from models.decoder_only_transformer import DecoderOnlyTransformer
import experiments.exp6_capstone as exp6

ckpt_path = 'checkpoints/exp6_capstone_seed0_step2000.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f"Step: {ckpt['step']}")
print(f"Config keys: {list(ckpt['config'].keys())}")
print(f"Model state dict keys: {len(ckpt['model_state_dict'])}")

# Recreate model from config (same logic as train_single_seed)
cfg = ckpt['config']
task_cfg = cfg['task']
model_cfg = cfg['model']

vocab_size = task_cfg['induction']['vocab_size'] + task_cfg['modular']['modulus'] + 10
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=model_cfg['d_model'],
    n_layers=model_cfg['n_layers'],
    n_heads=model_cfg['n_heads'],
    d_mlp=model_cfg['d_mlp'],
    dropout=model_cfg['dropout'],
    rotary_base=model_cfg['rotary_base'],
    rmsnorm_eps=model_cfg['rmsnorm_eps'],
).to('cpu')

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# Cell 2: Fourier analysis — k_99 trajectory, density confirmation
import numpy as np

# Compute Fourier decomposition on modular embedding slice
modular_offset = cfg['task']['modular'].get('vocab_offset', 0)
modulus = cfg['task']['modular']['modulus']

print(f"Modular offset: {modular_offset}, Modulus: {modulus}")

with torch.no_grad():
    embed_weights = model.embed.weight[modular_offset:modular_offset+modulus].detach().cpu().numpy()
    
fourier_result = exp6.fourier_decomposition(embed_weights, modulus)

k_99 = fourier_result['k_99']
k_90 = fourier_result['k_90']
sparsity = fourier_result['sparsity']

print(f"Fourier k_99: {k_99} / {modulus} ({k_99/modulus*100:.1f}%)")
print(f"Fourier k_90: {k_90} / {modulus}")
print(f"Final Fourier sparsity: {sparsity:.4f}")

# Density interpretation
if k_99 > modulus * 0.5:
    print(f"→ DENSE regime (k_99 > {modulus * 0.5:.0f}). Not the sparse Fourier circuit.")
else:
    print(f"→ SPARSE regime (k_99 <= {modulus * 0.5:.0f}). Grokking circuit detected.")

In [ ]:
# Cell 3: K-composition — adjacent-pair scoring, _vacuous marker
from experiments.exp1_induction_heads import make_repeated_token_data, compute_attention_entropy
from torch.utils.data import DataLoader, TensorDataset

# Create induction-only validation batches for K-comp
induction_cfg = cfg['task']['induction']

val_data = make_repeated_token_data(
    vocab_size=induction_cfg['vocab_size'],
    seq_len=induction_cfg['seq_len'],
    num_samples=induction_cfg['num_val'],
    seed=cfg.get('seed', 0) + 1  # different seed for val
)
val_loader = DataLoader(TensorDataset(val_data['input_ids'], val_data['target_ids']), 
                       batch_size=induction_cfg['batch_size'], shuffle=False)

# Compute K-composition on validation batch
batch = next(iter(val_loader))
input_ids, target_ids = batch[0], batch[1]

with torch.no_grad():
    kcomp_result = exp6.compute_k_composition_scores(model, input_ids, target_ids)

max_kcomp = kcomp_result['max_kcomp']
per_head_scores = kcomp_result['per_head_scores']
vacuous = kcomp_result.get('_vacuous', False)

print(f"Max K-comp: {max_kcomp:.4f}")
print(f"Per-head K-comp scores shape: {per_head_scores.shape}")
print(f"_vacuous marker: {vacuous}")

# Per-head diag+1 mass (the real detection threshold)
with torch.no_grad():
    attn_result = compute_attention_entropy(model, input_ids[:1])
    
diag1_mass = attn_result['diag1_mass']
max_diag1 = diag1_mass.max().item()
print(f"Max per-head diag+1 mass: {max_diag1:.4f} (threshold: 0.3)")

if max_diag1 >= 0.3:
    print("→ HEAD DETECTED (diag+1 mass >= 0.3)")
else:
    print("→ NO HEAD (diag+1 mass < 0.3). K-comp score is compositional, not per-head.")

if vacuous:
    print("→ K-comp reported as _vacuous: no confirmed head to attribute composition to.")

In [ ]:
# Cell 4: Circuit patching — activation patching + path patching with honest vacuous zeros
import experiments.exp4_circuit_patching as exp4

# Create clean and corrupted batches for patching
clean_data = make_repeated_token_data(
    vocab_size=induction_cfg['vocab_size'],
    seq_len=induction_cfg['seq_len'],
    num_samples=induction_cfg['batch_size'],
    seed=cfg.get('seed', 0)
)
corrupted_data = make_repeated_token_data(
    vocab_size=induction_cfg['vocab_size'],
    seq_len=induction_cfg['seq_len'],
    num_samples=induction_cfg['batch_size'],
    seed=cfg.get('seed', 0) + 2
)

clean_input_ids = clean_data['input_ids']
clean_target_ids = clean_data['target_ids']
corrupted_input_ids = corrupted_data['input_ids']
corrupted_target_ids = corrupted_data['target_ids']

# Run activation patching
print("Running activation patching...")
act_patch_result = exp4.run_activation_patching(
    model,
    clean_input_ids,
    clean_target_ids,
    corrupted_input_ids,
    corrupted_target_ids,
)

act_recovery = act_patch_result['mean_recovery']
print(f"Activation patching mean recovery: {act_recovery:.4f}")

# Run path patching (will report _vacuous if no head)
print("Running path patching...")
path_patch_result = exp4.run_path_patching_to_logits(
    model,
    clean_input_ids,
    clean_target_ids,
    corrupted_input_ids,
    corrupted_target_ids,
)

path_recovery = path_patch_result.get('mean_recovery', 0.0)
path_vacuous = path_patch_result.get('_vacuous', False)
print(f"Path patching mean recovery: {path_recovery:.4f}")
print(f"Path patching _vacuous: {path_vacuous}")

if path_vacuous:
    print("→ Path patching reports _vacuous: no confirmed head to isolate. Unit tests only.")
else:
    print("→ Path patching found direct head effect on logits.")

In [ ]:
# Cell 5: SAE features — harvest from step-2000 checkpoint
import experiments.exp5_sae_dashboard as exp5

# Harvest real activations from ln_final
print("Harvesting activations from ln_final...")
activations = exp6.harvest_activations(model, num_batches=50, batch_size=32)
print(f"Activations shape: {activations.shape}")

# Train SAE on real activations
print("Training SAE on real activations...")
sae_result = exp5.train_sae(
    activations,
    dict_size=256,
    lr=1e-3,
    steps=5000,
    device='cpu'
)

fve = sae_result['fve']
l0 = sae_result['l0']
dead = sae_result['dead_features']

print(f"SAE on real activations (step-2000 checkpoint):")
print(f"  FVE: {fve:.4f} ({fve*100:.2f}%)")
print(f"  L0: {l0:.1f} / 256 ({l0/256*100:.1f}% active)")
print(f"  Dead features: {dead} / 256")

# Compare with synthetic baseline
print("\nComparison with synthetic baseline (RESULTS.md):")
print(f"  Synthetic: FVE 97.5%, L0 96.6/512 (18.9%), 39/512 dead")
print(f"  Real (step-2000): FVE {fve*100:.2f}%, L0 {l0:.1f}/256 ({l0/256*100:.1f}%), {dead}/256 dead")

if l0/256 > 0.3:
    print("→ SPARSITY GAP: Real activations not sparse. Small undertrained residual stream.")
else:
    print("→ Real activations show sparse features.")

# Cell 6: Literature comparison

## Comparison with Primary Literature

| Paper | Finding | This Run |
|-------|---------|----------|
| Nanda et al. 2023 (Progress Measures) | Sparse Fourier circuit at grokking (k_99 < P/2) | **DENSE** (k_99 = 98.1/113). Val accuracy 1.0 in standalone P=113 run, but joint training shows modular at chance. |
| Olsson et al. 2022 (Induction Heads) | Induction heads emerge at ~0.3 diag+1 mass | **NO HEAD** (max diag+1 < 0.3). Induction accuracy 0.50 via composition, not single head. |
| Elhage et al. 2022 (Superposition) | Phase transition: monosemantic → superposed | **REPRODUCED** (Rung 3): 10/20 → 20/20 features, pentagon geometry. |
| Wang et al. 2023 (IOI Circuit) | Activation/path patching recovers circuit | **ACTIVE**: Activation patching ~0.20 recovery. Path patching _vacuous (no head). |
| Bricken et al. 2023 (SAE) | Sparse features from real activations | **DENSE RECONSTRUCTION**: 99.97% FVE but 53% L0. Not yet sparse.

---

## Key Discrepancies

1. **No sparse Fourier at P=113** — All runs in this repo (P=29, 59, 113, CPU and joint) produce dense Fourier. The canonical grokking phase transition (sparse circuit) is not reproduced.
2. **Induction without heads** — Induction accuracy reaches 0.50 but no single head crosses 0.3 diag+1 threshold. K-comp detects composition, not a head.
3. **SAE sparsity gap** — Real activations reconstruct better but far less sparsely than synthetic. The residual stream may not contain disentangled features yet.
4. **Joint training dissociation** — Modular and induction tasks compete; induction wins. Modular never leaves chance in 2000 steps.

# Cell 7: Honest conclusion

## Honest Conclusion

> **This config finds a dense algorithm for modular addition and compositional induction without a confirmed head. The dissociation is the finding.**

### What we know:
- The MP-88 shakedown (2000 steps, joint training) produces a clean dissociation: induction accuracy rises to 0.504 while modular stays at chance (0.0047) with dense Fourier (k_99 = 98.1).
- The MP-89 retune A/B (500 steps) falsifies both standing hypotheses: dedicated vocab-offset (interference) and curriculum reweight (schedule) do not move modular off chance.
- Rung 3 (superposition) is the strongest verified result: clean phase transition with pentagon geometry.
- Rung 2 (grokking) is a positive-negative: the model solves modular addition (val 1.0) but via dense algorithm, not sparse Fourier.
- Activation patching finds real circuit sensitivity (~0.20 recovery) but no head crosses the detection threshold.
- SAE on the shakedown checkpoint reconstructs densely (53% L0), not sparsely.

### What we don't claim:
- **No induction head.** K-comp 0.394 is a composition score, not per-head diag+1 mass.
- **No modular mechanism.** Dense Fourier at k_99 98.1 matches the NO-GROK baseline.
- **Below-chance modular is a footnote.** 0.0047 vs 0.0088 chance is consistent with interference but not evidence.

### Next research question (if any):
Gated on GPU run producing sparse Fourier, or a new candidate set (architecture, optimizer, data, scale) in the next continuum ledger. The dense attractor under joint training is characterized as the contribution.